In [ ]:
!pip3 install -q torchmetrics
!pip3 install -q segmentation-models-pytorch
!pip3 install -q albumentations

# Libraries

In [ ]:
import os
import json
import multiprocessing
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageColor

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import torchmetrics
from torchmetrics import Dice, JaccardIndex

import segmentation_models_pytorch as smp
from tqdm import tqdm

In [ ]:
print(f"PyTorch version: {torch.__version__}")

# Check for available devices
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"

print(f"Using device: {device}")

# Create data and send it to the device
x = torch.rand(size=(3, 4)).to(device)
print(x)

# Informations

In [ ]:
### Important paths
if os.name == 'posix':
    if os.uname().sysname == 'Linux':
        project_dir = Path("/kaggle/input") 
    elif os.uname().sysname == 'Darwin':
        project_dir = Path("/Users/lexuanthang/Library/CloudStorage/OneDrive-Personal/01_WORKING/02_cell_detection")
else:
    raise EnvironmentError("Unsupported operating system")
### Important paths

data_dir = project_dir / "cell-detection"
images_dir = data_dir / "P01_images"
masks_dir = data_dir / "P01_masks"

print("input image path: {} \ninput mask path: {}".format(images_dir, masks_dir))

# validate_image_mask_pairs(images_dir, masks_dir)

# Define the classes
CELL_CLASSES = [
    # "Background",
    "Macrophage/Monocyte",
    "Neutrophil",
    "Eosinophil",
    "Lymphocyte",
    "Unknown cell/Debris",
]
NUM_CLASSES = len(CELL_CLASSES) + 1 
print("number of classes: {}".format(NUM_CLASSES)
      )
trainsize = 256
train_transform = A.Compose([
    A.Resize(width=trainsize, height=trainsize),
    A.HorizontalFlip(),
    # A.RandomBrightnessContrast(),
    A.Blur(),
    A.Sharpen(),
    # A.RGBShift(),
    A.CoarseDropout(),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0),
    ToTensorV2(),
])

# Define color to class index mapping
label_colors = {
    # (0, 0, 0): 0,  # Background
    (28, 230, 255): 1,  # Macrophage/Monocyte
    (255, 52, 255): 2,  # Neutrophil
    (255, 74, 70): 3,  # Eosinophil
    (0, 137, 65): 4,  # Lymphocyte
    (0, 111, 166): 5,  # Unknown cell/Debris
    # (163, 0, 89): 6   # Basophil
}

# Dataset

In [ ]:
class CellDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        """
        Args:
            images_dir (string): Path to the directory containing images.
            masks_dir (string): Path to the directory containing corresponding masks.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform
        self.images = sorted(os.listdir(images_dir))
        self.masks = sorted(os.listdir(masks_dir))
        
        # Create a dictionary to map image names to mask names
        self.image_mask_map = {img_name: self.find_matching_mask(img_name) for img_name in self.images}

    def find_matching_mask(self, img_name):
        # Assuming the mask file has the same name as the image file but with a different prefix
        mask_name = img_name.replace("P01_", "masks_P01_")
        if mask_name in self.masks:
            return mask_name
        else:
            raise ValueError(f"No matching mask found for image {img_name}")

    def __len__(self):
        """
        Return the total number of samples.
        """
        return len(self.images)

    def __getitem__(self, idx):
        """
        Generate one sample of data.
        """
        img_name = self.images[idx]
        mask_name = self.image_mask_map[img_name]
        img_path = os.path.join(self.images_dir, img_name)
        mask_path = os.path.join(self.masks_dir, mask_name)

        image = Image.open(img_path).convert('RGB')
        image = np.array(image)
        mask = Image.open(mask_path).convert('RGB')

        # convert mask colors to classes
        mask = self.convert_mask(mask)

        if self.transform is not None:
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]
        # print(f"Loaded image: {img_name}, mask: {mask_name}")  # Debug print
        return image.float(), mask.long()

    def convert_mask(self, mask):
        """Convert RGB mask to a class map."""
        mask_array = np.array(mask)
        class_map = np.zeros(mask_array.shape[:2], dtype=np.int32)

        for color, class_id in label_colors.items():
            matches = (mask_array == color).all(axis=-1)
            class_map[matches] = class_id

        return class_map

In [ ]:
# Initialize dataset
dataset = CellDataset(images_dir,masks_dir,transform=train_transform)
image,mask = dataset.__getitem__(1)
print(image.shape, mask.shape)
print(image.unique(), mask.unique())
plt.subplot(1,2,1)
plt.imshow(image.permute(1,2,0).cpu())
plt.subplot(1,2,2)
plt.imshow(mask.cpu())
plt.show()

In [ ]:
def tensor_to_np(tensor):
    # Make sure the tensor is on the CPU and convert to NumPy
    return tensor.detach().cpu().numpy()

def np_to_tensor(array):
    # Convert a NumPy array back to PyTorch tensor
    return torch.tensor(array).float()

def inverse_norm(image):
    # Define the inverse transformation using Albumentations
    invTrans = A.Compose([
        A.Normalize(mean=[0., 0., 0.], std=[1/0.229, 1/0.224, 1/0.225], max_pixel_value=1.0),
        A.Normalize(mean=[-0.485, -0.456, -0.406], std=[1., 1., 1.], max_pixel_value=1.0),
    ])

    # Example usage:
    # Assuming 'tensor_image' is your normalized image tensor
    tensor_image_np = tensor_to_np(image)  # Convert tensor to numpy array
    tensor_image_np = np.transpose(tensor_image_np, (1, 2, 0))  # CHW to HWC for Albumentations

    # Apply the inverse transformation
    inv_img_np = invTrans(image=tensor_image_np)['image']
    inv_img_np = np.transpose(inv_img_np, (2, 0, 1))  # HWC back to CHW for PyTorch

    # Convert back to tensor
    inv_img_tensor = np_to_tensor(inv_img_np)
    return inv_img_tensor


In [ ]:
train_dataset = CellDataset(images_dir,masks_dir,transform=train_transform)
image, mask = train_dataset.__getitem__(10)

print(image.shape, mask.shape)
print(mask.unique())
inv_img_tensor = inverse_norm(image)
print(inv_img_tensor)
plt.subplot(1,2,1)
plt.imshow(inv_img_tensor.permute(1,2,0))
plt.subplot(1,2,2)
plt.imshow(mask)
plt.show()

In [ ]:
from torch.utils.data import random_split
# Initialize dataset
full_dataset = CellDataset(images_dir,masks_dir,transform=train_transform)

# split dataset
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
tes_size = len(full_dataset) - train_size - val_size
print("train size: {}, val size: {}, test size: {}".format(train_size, val_size, tes_size))
train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, tes_size])
print("train size: {}, val size: {}, test size: {}".format(len(train_dataset), len(val_dataset), len(test_dataset)))

In [ ]:
BATCH_SIZE = 16
NUM_WORKERS = multiprocessing.cpu_count()

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Defining the Models


## Models

In [ ]:
import segmentation_models_pytorch as smp

UNet = smp.Unet(
    encoder_name="resnet50",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
    encoder_weights="imagenet",     # use `imagenet` pre-trained weights for encoder initialization
    in_channels=3,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
    classes=NUM_CLASSES,                      # model output channels (number of classes in your dataset)
)

PSPNet = smp.PSPNet(
    encoder_name="resnet50",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
    encoder_weights="imagenet",     # use `imagenet` pre-trained weights for encoder initialization
    in_channels=3,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
    classes=NUM_CLASSES,                      # model output channels (number of classes in your dataset)
)

DeepLabV3 = smp.DeepLabV3(
    encoder_name="resnet50",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
    encoder_weights="imagenet",     # use `imagenet` pre-trained weights for encoder initialization
    in_channels=3,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
    classes=NUM_CLASSES,                      # model output channels (number of classes in your dataset)
)


In [ ]:
# Test images in model
image_size = 256
x = torch.rand(4,3,image_size, image_size).float()
y = torch.ones(4,image_size,image_size).long()

a1 = UNet(x)
print(a1.shape)
a1 = PSPNet(x)
print(a1.shape)
a1 = DeepLabV3(x)
print(a1.shape)

# Training models

## Metrics

In [ ]:
class AverageMetric(object):
    def __init__(self):
        self.reset()

    def reset(self):
        self.val=0
        self.avg=0
        self.sum=0
        self.count=0

    def update(self, val, n=1):
        self.val = val
        self.sum += val*n
        self.count += n
        self.avg = self.sum / self.count


def intersectionAndUnionGPU(output, target, K, ignore_index=255):
    # 'K' classes, output and target sizes are N or N * L or N * H * W, each value in range 0 to K - 1.
    assert (output.dim() in [1, 2, 3])
    assert output.shape == target.shape
    output = output.view(-1)
    target = target.view(-1)
    output[target == ignore_index] = ignore_index
    intersection = output[output == target]
    area_intersection = torch.histc(intersection, bins=K, min=0, max=K-1)
    area_output = torch.histc(output, bins=K, min=0, max=K-1)
    area_target = torch.histc(target, bins=K, min=0, max=K-1)
    area_union = area_output + area_target - area_intersection
    return area_intersection, area_union, area_target

#metrics
dice_fn = torchmetrics.Dice(num_classes=NUM_CLASSES, average="macro").to(device)
# iou_fn = intersectionAndUnionGPU().to(device)
acc_fn = torchmetrics.Accuracy(num_classes=NUM_CLASSES, task="multiclass").to(device)

# metric
acc_metric = AverageMetric()
dice_metric = AverageMetric()
intersection_metric = AverageMetric()
union_metric = AverageMetric()
target_metric = AverageMetric()
train_loss_metric = AverageMetric()


In [ ]:
import torch
import torchmetrics
from sklearn.metrics import f1_score

# Các hàm tính toán các chỉ số
def boundary_f1_score(pred, target, threshold=0.5):
    """Tính toán Boundary F1 score giữa dự đoán và thực tế."""
    # Biến đổi dự đoán và ground truth thành các biên
    pred_boundaries = (pred > threshold).float()
    target_boundaries = (target > threshold).float()

    # Tính Precision và Recall cho các biên
    intersection = (pred_boundaries * target_boundaries).sum()
    precision = intersection / (pred_boundaries.sum() + 1e-10)
    recall = intersection / (target_boundaries.sum() + 1e-10)

    # Tính F1 Score
    f1 = 2 * (precision * recall) / (precision + recall + 1e-10)
    return f1

def pixel_accuracy(pred, target):
    """Tính toán pixel accuracy giữa dự đoán và thực tế."""
    correct = (pred == target).sum().float()
    total = target.numel()
    return correct / total

def f1_per_class(pred, target, num_classes):
    """Tính F1 Score cho từng lớp."""
    f1_scores = []
    for i in range(num_classes):
        pred_class = (pred == i).float()
        target_class = (target == i).float()

        # Tính precision và recall
        intersection = (pred_class * target_class).sum()
        precision = intersection / (pred_class.sum() + 1e-10)
        recall = intersection / (target_class.sum() + 1e-10)

        # Tính F1 Score cho lớp i
        f1 = 2 * (precision * recall) / (precision + recall + 1e-10)
        f1_scores.append(f1)
    return f1_scores


def compute_metrics(yhat_mask, y, num_classes):
    intersection, union, target = intersectionAndUnionGPU(yhat_mask.float(), y.float(), num_classes)
    accuracy = acc_fn(yhat_mask, y.long())

    # Tính các chỉ số bổ sung
    boundary_f1 = boundary_f1_score(yhat_mask, y)
    pixel_acc = pixel_accuracy(yhat_mask, y)
    class_f1 = f1_per_class(yhat_mask, y, num_classes)

    intersection_metric.update(intersection.sum().item())
    union_metric.update(union.sum().item())
    target_metric.update(target.sum().item())

    return accuracy, boundary_f1, pixel_acc, class_f1

## Train Epochs

In [ ]:
def train_epoch(model, train_loader, val_loader,
                optimizer, criterion, device, num_classes, 
                early_stopping, model_name, epoch):
    # Initialize dictionaries to store metrics for training and validation
    metrics = {
        'train': {
            'train_loss': [],
            'train_accuracy': [],
            'train_mIoU': [],
            'train_mDice': [],
            'train_boundary_f1': [],
            'train_pixel_accuracy': []
        },
        'val': {
            'val_loss': [],
            'val_accuracy': [],
            'val_mIoU': [],
            'val_mDice': [],
            'val_boundary_f1': [],
            'val_pixel_accuracy': []
        }
    }

    # Training phase
    model.train()
    for batch_id, (x, y) in enumerate(tqdm(train_loader)):
        optimizer.zero_grad()
        n = x.shape[0]

        x = x.to(device).float()
        y = y.to(device).long()

        yhat_mask = model(x)  # Model prediction

        # Compute training loss
        main_loss = criterion(yhat_mask, y)
        loss = main_loss

        # Backpropagation
        loss.backward()
        optimizer.step()

        # Compute metrics
        with torch.no_grad():
            yhat_mask = yhat_mask.argmax(dim=1).squeeze()  # Convert to class indices
            accuracy, boundary_f1, pixel_acc, class_f1 = compute_metrics(yhat_mask, y, num_classes)

            # Update training metrics
            metrics['train']['train_loss'].append(loss.item())
            metrics['train']['train_accuracy'].append(accuracy.item())
            metrics['train']['train_boundary_f1'].append(boundary_f1.item())
            metrics['train']['train_pixel_accuracy'].append(pixel_acc.item())

            # Compute mIoU, mDice for training
            intersection_metric = AverageMetric()
            union_metric = AverageMetric()
            target_metric = AverageMetric()

            intersection, union, target = intersectionAndUnionGPU(yhat_mask.float(), y.float(), num_classes)

            intersection_metric.update(intersection.sum().item())
            union_metric.update(union.sum().item())
            target_metric.update(target.sum().item())

            iou_class = intersection_metric.sum / (union_metric.sum + 1e-10)
            dice_class = (2 * intersection_metric.sum) / (intersection_metric.sum + union_metric.sum + 1e-10)

            mIoU = torch.mean(torch.tensor(iou_class))
            mDice = torch.mean(torch.tensor(dice_class))

            # Update training mIoU and mDice
            metrics['train']['train_mIoU'].append(mIoU.item())
            metrics['train']['train_mDice'].append(mDice.item())

    # Validation phase (evaluate model on validation data)
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0
    model.eval()  # Switch model to evaluation mode

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device).float()
            y = y.to(device).long()

            yhat_mask = model(x)

            # Compute validation loss
            val_loss = criterion(yhat_mask, y)
            val_loss_sum += val_loss.item()

            # Compute accuracy and mIoU
            yhat_mask = yhat_mask.argmax(dim=1).squeeze()  # Convert to class indices
            accuracy, boundary_f1, pixel_acc, class_f1 = compute_metrics(yhat_mask, y, num_classes)

            # Update validation metrics
            metrics['val']['val_loss'].append(val_loss.item())
            metrics['val']['val_accuracy'].append(accuracy.item())
            metrics['val']['val_boundary_f1'].append(boundary_f1.item())
            metrics['val']['val_pixel_accuracy'].append(pixel_acc.item())

            # Compute mIoU, mDice for validation
            intersection_metric = AverageMetric()
            union_metric = AverageMetric()
            target_metric = AverageMetric()

            intersection, union, target = intersectionAndUnionGPU(yhat_mask.float(), y.float(), num_classes)

            intersection_metric.update(intersection.sum().item())
            union_metric.update(union.sum().item())
            target_metric.update(target.sum().item())

            iou_class = intersection_metric.sum / (union_metric.sum + 1e-10)
            dice_class = (2 * intersection_metric.sum) / (intersection_metric.sum + union_metric.sum + 1e-10)

            mIoU = torch.mean(torch.tensor(iou_class))
            mDice = torch.mean(torch.tensor(dice_class))

            # Update validation mIoU and mDice
            metrics['val']['val_mIoU'].append(mIoU.item())
            metrics['val']['val_mDice'].append(mDice.item())

    # Compute average validation loss
    val_loss_avg = val_loss_sum / len(val_loader)

    # Return all metrics
    return metrics

## Early Stopping

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, verbose=False, delta=0, path="checkpoint.pth"):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.best_metric = None
        self.best_epoch = 0
        self.counter = 0
        self.early_stop = False
        self.path = path

    def __call__(self, metric, model, model_name):
        if self.best_metric is None:
            self.best_metric = metric
            self.save_checkpoint(model, model_name)
        elif metric < self.best_metric - self.delta:
            self.counter += 1
            print(f"EarlyStopping counter for {model_name}: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_metric = metric
            self.save_checkpoint(model, model_name)
            self.counter = 0

    def save_checkpoint(self, model, model_name):
        if self.verbose:
            print(f"Validation metric improved for {model_name}. Saving model...")
        torch.save(model.state_dict(), f"{model_name}_best_model.pth")

    def load_checkpoint(self, model, model_name):
        model.load_state_dict(torch.load(f"{model_name}_best_model.pth"))
        print(f"Checkpoint for {model_name} loaded.")

## Train models

In [ ]:
# Assuming you have multiple GPUs, if you have more than one GPU, we can use DataParallel
# This will utilize all available GPUs
def prepare_model_for_multiple_gpus(model):
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = torch.nn.DataParallel(model)  # Wrap the model with DataParallel
    model = model.to(device)
    return model


def train_models(models, train_loader, val_loader, 
                 criterion, num_epochs, device, num_classes, 
                 early_stopping_dict):
    
    results = {}  # Dictionary to store the results for each model
    
    for model_name, model in models.items():
        print(f"*** Training model: {model_name} ***")
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        model = model.to(device)

        early_stopping = early_stopping_dict.get(model_name)

        model_results = {
            'train_loss': [],
            'train_accuracy': [],
            'train_mIoU': [],
            'train_mDice': [],
            'train_boundary_f1': [],
            'train_pixel_accuracy': [],
            'val_loss': [],
            'val_accuracy': [],
            'val_mIoU': [],
            'val_mDice': [],
            'val_boundary_f1': [],
            'val_pixel_accuracy': []
        }

        best_val_mIoU = -float('inf')  # Initialize to very low value
        for epoch in range(num_epochs):
            print(f"Epoch {epoch + 1}/{num_epochs}")

            # Get the metrics dictionary for this epoch
            metrics = train_epoch(
                model, train_loader, val_loader, optimizer, criterion, device, num_classes, early_stopping, model_name, epoch + 1
            )

            # Store metrics for this epoch
            model_results['train_loss'].append(metrics['train']['train_loss'][-1])
            model_results['train_accuracy'].append(metrics['train']['train_accuracy'][-1])
            model_results['train_mIoU'].append(metrics['train']['train_mIoU'][-1])
            model_results['train_mDice'].append(metrics['train']['train_mDice'][-1])
            model_results['train_boundary_f1'].append(metrics['train']['train_boundary_f1'][-1])
            model_results['train_pixel_accuracy'].append(metrics['train']['train_pixel_accuracy'][-1])

            model_results['val_loss'].append(metrics['val']['val_loss'][-1])
            model_results['val_accuracy'].append(metrics['val']['val_accuracy'][-1])
            model_results['val_mIoU'].append(metrics['val']['val_mIoU'][-1])
            model_results['val_mDice'].append(metrics['val']['val_mDice'][-1])
            model_results['val_boundary_f1'].append(metrics['val']['val_boundary_f1'][-1])
            model_results['val_pixel_accuracy'].append(metrics['val']['val_pixel_accuracy'][-1])

            # Print metrics for this epoch
            print(f"Epoch {epoch + 1} Metrics for {model_name}:")
            
            print(f"  Training Loss: {metrics['train']['train_loss'][-1]:.4f}, "
                  f"Training Accuracy: {metrics['train']['train_accuracy'][-1]:.4f}, "
                  f"Training mIoU: {metrics['train']['train_mIoU'][-1]:.4f}, "
                  f"Training mDice: {metrics['train']['train_mDice'][-1]:.4f}, "
                  f"Training Boundary F1: {metrics['train']['train_boundary_f1'][-1]:.4f}, "
                  f"Training Pixel Accuracy: {metrics['train']['train_pixel_accuracy'][-1]:.4f}")

            print(f"  Validation Loss: {metrics['val']['val_loss'][-1]:.4f}, "
                  f"Validation Accuracy: {metrics['val']['val_accuracy'][-1]:.4f}, "
                  f"Validation mIoU: {metrics['val']['val_mIoU'][-1]:.4f}, "
                  f"Validation mDice: {metrics['val']['val_mDice'][-1]:.4f}, "
                  f"Validation Boundary F1: {metrics['val']['val_boundary_f1'][-1]:.4f}, "
                  f"Validation Pixel Accuracy: {metrics['val']['val_pixel_accuracy'][-1]:.4f}")

            # Early stopping check
            if early_stopping and early_stopping.early_stop:
                print(f"Early stopping for {model_name}")
                break

            # Save model if validation mIoU improves
            if metrics['val']['val_mIoU'][-1] > best_val_mIoU:
                print(f"Validation metric improved for {model_name}. Saving model...")
                best_val_mIoU = metrics['val']['val_mIoU'][-1]
                torch.save(model.state_dict(), f"{model_name}_best_model.pth")

        # Store results for the model
        results[model_name] = model_results

        # Save the results to a JSON file
        with open(f'{model_name}_results.json', 'w') as f:
            json.dump(model_results, f)

    return results

In [ ]:

# Đoạn code huấn luyện 3 mô hình với các kết quả theo từng epoch
device = "cuda" if torch.cuda.is_available() else "cpu"
num_classes = NUM_CLASSES  # Modify based on your problem

# Khởi tạo các mô hình
models = {
    "UNet": smp.Unet(
        encoder_name="resnet50", 
        encoder_weights="imagenet", 
        in_channels=3, 
        classes=num_classes
    ).to(device),
    
    "PSPNet": smp.PSPNet(
        encoder_name="resnet50", 
        encoder_weights="imagenet", 
        in_channels=3, 
        classes=num_classes
    ).to(device),
    
    "DeepLabV3": smp.DeepLabV3(
        encoder_name="resnet50", 
        encoder_weights="imagenet", 
        in_channels=3, 
        classes=num_classes
    ).to(device)
}

criterion = torch.nn.CrossEntropyLoss()
num_epochs = 100

# Initialize EarlyStopping for each model
early_stopping_unet = EarlyStopping(patience=3, verbose=True, path="UNet_best_model.pth")
early_stopping_pspnet = EarlyStopping(patience=3, verbose=True, path="PSPNet_best_model.pth")
early_stopping_deeplabv3 = EarlyStopping(patience=3, verbose=True, path="DeepLabV3_best_model.pth")

early_stopping_dict = {
    "UNet": early_stopping_unet,
    "PSPNet": early_stopping_pspnet,
    "DeepLabV3": early_stopping_deeplabv3
}

# Huấn luyện mô hình và theo dõi các chỉ số
results = train_models(models, train_loader, val_loader,
                       criterion, num_epochs, device, num_classes,
                       early_stopping_dict)


# Check results

In [ ]:
import torch
import matplotlib.pyplot as plt
import json
import os

def load_model(model_dir, model_name, model_class, device):
    """Load the model from the saved model file."""
    model = model_class.to(device)  # Initialize the model with the same architecture
    model.load_state_dict(torch.load(f"{model_dir}/{model_name}_best_model.pth"))
    model.eval()  # Switch to evaluation mode
    return model

def load_results(model_dir,model_name):
    """Load the results (metrics) from the saved JSON file."""
    with open(f'{model_dir}/{model_name}_results.json', 'r') as f:
        results = json.load(f)
    return results

def plot_comparison_metrics(results):
    """Plot comparison of each metric for all models in separate plots."""
    # Number of models
    models = list(results.keys())

    # Number of metrics
    metrics = [
        ('train_loss', 'Training Loss'),
        ('train_accuracy', 'Training Accuracy'),
        ('train_mIoU', 'Training mIoU'),
        ('train_mDice', 'Training mDice'),
        ('val_loss', 'Validation Loss'),
        ('val_accuracy', 'Validation Accuracy'),
        ('val_mIoU', 'Validation mIoU'),
        ('val_mDice', 'Validation mDice')
    ]
    
    # Create a plot for each metric
    plt.figure(figsize=(6, 8.9))

    for i, (metric_name, plot_title) in enumerate(metrics, 1):
        plt.subplot(4, 2, i)  # Create a 4x2 grid of plots
        for model in models:
            plt.plot(range(1, len(results[model][metric_name]) + 1), results[model][metric_name], label=f"{model}")
        plt.title(plot_title)
        plt.xlabel('Epochs')
        plt.ylabel(plot_title.split()[1])  # "Loss", "Accuracy", "IoU", etc.
        plt.legend()

    # Adjust layout to prevent overlap of plots
    plt.tight_layout()
    plt.show()

def load_and_plot_all_models(model_dir, models, device):
    """Load all models, their results, and plot the comparison of metrics."""
    results = {}

    # Load each model and its results
    for model_name, model_class in models.items():
        print(f"Loading and plotting for model: {model_name}")
        model = load_model(model_dir,model_name, model_class, device)
        model_results = load_results(model_dir,model_name)
        results[model_name] = model_results

    # Plot comparison of metrics for all models
    plot_comparison_metrics(results)

In [ ]:
# Define your models (change the model architecture as needed)
models = {
    "UNet": smp.Unet(encoder_name="resnet50", encoder_weights="imagenet", in_channels=3, classes=NUM_CLASSES),
    "PSPNet": smp.PSPNet(encoder_name="resnet50", encoder_weights="imagenet", in_channels=3, classes=NUM_CLASSES),
    "DeepLabV3": smp.DeepLabV3(encoder_name="resnet50", encoder_weights="imagenet", in_channels=3, classes=NUM_CLASSES)
}
device = "cuda" if torch.cuda.is_available() else "cpu"
model_dir = "/kaggle/input/paper-2025-benchmarkdataset-results"
load_and_plot_all_models(model_dir, models, device)

## Test_Loader

In [ ]:
def evaluate_model_with_loader(model, test_loader, device, num_classes):
    """Evaluates the model on the test loader and prints the final results."""
    model.eval()
    acc_metric.reset()
    dice_metric.reset()
    intersection_metric.reset()
    union_metric.reset()
    target_metric.reset()

    with torch.no_grad():
        for x, y in tqdm(test_loader, desc="Evaluating Test Set"):
            x = x.to(device).float()
            y = y.to(device).long()

            yhat_mask = model(x)
            yhat_mask_argmax = yhat_mask.argmax(dim=1).squeeze()

            accuracy, boundary_f1, pixel_acc, class_f1 = compute_metrics(yhat_mask_argmax, y, num_classes)
            acc_metric.update(accuracy.item())

            dice = dice_fn(yhat_mask, y)
            dice_metric.update(dice.item())

    iou_class = intersection_metric.sum / (union_metric.sum + 1e-10)
    mIoU = torch.mean(torch.tensor(iou_class)).item()
    mDice = dice_metric.avg
    accuracy_avg = acc_metric.avg

    print("\n--- Test Set Final Results ---")
    print(f"Test Accuracy: {accuracy_avg:.4f}, \n"
          f"Test mIoU: {mIoU:.4f}, \n"
          f"Test mDice: {mDice:.4f}, \n"
          f"Test Boundary F1: {boundary_f1:.4f}, \n"
          f"Test Pixel Accuracy: {pixel_acc:.4f} \n")

def Load_and_eval_test_loader_models(model_dir, models, test_loader, device, NUM_CLASSES):
    """Load all models, their results, and plot the comparison of metrics."""

    # Load each model and its results
    for model_name, model_class in models.items():
        print(f"Loading and Evaluating for model: {model_name}")
        model = load_model(model_dir,model_name, model_class, device)
        # Evaluate the loaded model on the test loader
        evaluate_model_with_loader(model, test_loader, device, NUM_CLASSES)
        
Load_and_eval_test_loader_models(model_dir, models, test_loader, device, NUM_CLASSES)

In [ ]:
def visualization_pascalvoc(model, idx, train_dataset, device):
    model.eval()
    
    # Get image and label from the dataset
    with torch.no_grad():
        x, y = train_dataset[idx]  # Get x: (C, H, W), y: (H, W)

        # Prepare the input and target for the model (batch dimension is added)
        x = x.to(device).float().unsqueeze(0)  # x: (C, H, W) -> (1, C, H, W)
        y = y.to(device).long().unsqueeze(0)  # y: (H, W) -> (1, H, W) 
        
        # Perform inference
        yhat = model(x)  # model output yhat: (1, C, H, W)
        
        # Get the predicted class for each pixel
        yhat_mask = yhat.argmax(dim=1)  # yhat_mask: (1, H, W)

        # Inverse normalization of the image (if needed, adjust the inverse normalization process)
        inv_img_tensor = inverse_norm(x.squeeze())  # Reverse the normalization (if done earlier)
        
        # Visualization
        # Plot the original image, ground truth, and predicted mask side by side
        plt.figure(figsize=(12, 4))
        
        # Original Image
        plt.subplot(1, 3, 1)
        plt.imshow(inv_img_tensor.permute(1, 2, 0).cpu())  # Convert (C, H, W) to (H, W, C)
        plt.title("Original Image")
        plt.axis('off')
        
        # Ground truth mask
        plt.subplot(1, 3, 2)
        plt.imshow(y.squeeze().cpu(), cmap='tab10')  # Display ground truth using tab10 colormap
        plt.title("Ground Truth")
        plt.axis('off')
        
        # Predicted mask
        plt.subplot(1, 3, 3)
        plt.imshow(yhat_mask.squeeze().cpu(), cmap='tab10')  # Display predicted mask using tab10 colormap
        plt.title("Predicted Mask")
        plt.axis('off')

        plt.show()

import random
idx = random.randint(0,90)

visualization_pascalvoc(models['UNet'], idx, train_dataset, device)

In [ ]:
!zip output.zip *pth *json